In [ ]:
import csv
import json
import random
import sys
from pathlib import Path
from typing import Any

import numpy as np
import torch

In [ ]:
from flowrec.data.datamodule import AmazonReviewDataConfig, AmazonReviewDataModule
from flowrec.models.popularity import PopularityConfig, PopularityModel
from flowrec.models.svd import SVDConfig, SVDModel


In [ ]:
SEEDS = (42, 43, 44)

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

In [ ]:
AMAZON_DATASETS = {
    "beauty": {
        "reviews_path": "../data/raw/reviews_Beauty.json.gz",
        "metadata_path": "../data/raw/meta_Beauty.json.gz",
        "item_key": "asin",
    },

    "instruments": {
        "reviews_path": "../data/raw/Musical_Instruments.json.gz",
        "metadata_path": "../data/raw/meta_Musical_Instruments.json.gz",
        "item_key": "asin",
    },
}


In [ ]:
def make_data_config(dataset_cfg: dict[str, Any], seed: int) -> AmazonReviewDataConfig:
    return AmazonReviewDataConfig(
        reviews_path=dataset_cfg["reviews_path"],
        metadata_path=dataset_cfg.get("metadata_path"),
        item_key=dataset_cfg.get("item_key", "parent_asin"),

        # Global time split.
        split_strategy="leave_one_out",
        global_history_policy="rolling",
        global_eval_target_mode="last_per_user",

        # Requested config.
        train_all_prefixes=True,
        min_user_interactions=5,
        min_item_interactions=5,
        max_history_len=20,
        deduplicate_consecutive=True,

        # Batching.
        batch_size=256,
        eval_batch_size=256,
        num_workers=0,
        seed=seed,
    )

In [ ]:
def fit_popularity_model(dm) -> PopularityModel:
    num_items = dm.item_vocab_size

    model = PopularityModel(
        PopularityConfig(
            num_items=num_items,
            pad_item_id=0,
            score_transform="log1p",
            exclude_history_in_generate=False,
        )
    )

    store = getattr(dm, "store", None)
    splits = getattr(dm, "splits", None)
    train_index = getattr(splits, "train", None) if splits is not None else None

    if store is not None and train_index is not None and hasattr(model, "fit_from_store"):
        try:
            model.fit_from_store(
                store=store,
                target_index=train_index,
                count_targets_only=False,
            )
        except TypeError:
            model.fit_from_store(
                store=store,
                target_index=train_index,
                count_targets=True,
                count_history=True,
            )
    else:
        model.fit_from_dataloader(
            dm.train_dataloader(),
            count_targets=True,
            count_history=True,
        )

    return model


def fit_svd_model(dm) -> SVDModel:
    num_users = dm.num_users
    num_items = dm.item_vocab_size

    model = SVDModel(
        SVDConfig(
            num_users=num_users,
            num_items=num_items,
            n_components=128,
            input_transform="binary",
            user_representation="history_mean",
            foldin_reg=1e-2,
            exclude_history_in_generate=False,
            )
    )

    store = getattr(dm, "store", None)
    splits = getattr(dm, "splits", None)
    train_index = getattr(splits, "train", None) if splits is not None else None

    if store is not None and train_index is not None and hasattr(model, "fit_from_store"):
        model.fit_from_store(
            store=store,
            target_index=train_index,
            fit_source="observed_prefix",
            use_ratings=False,
        )
    else:
        model.fit_from_dataloader(
            dm.train_dataloader(),
            count_history=True,
            count_targets=True,
            use_ratings=False,
        )

    return model

In [ ]:
@torch.no_grad()
def evaluate_model(
    model,
    test_loader,
    *,
    num_items: int,
    ks: tuple[int, ...] = (5, 10, 20),
    pad_item_id: int = 0,
) -> dict[str, float]:
    model.eval()

    max_k = max(ks)

    total = 0
    hit_sum = {k: 0.0 for k in ks}
    ndcg_sum = {k: 0.0 for k in ks}
    coverage_mask = {k: torch.zeros(num_items, dtype=torch.bool) for k in ks}

    for batch in test_loader:
        target = batch.get("target_item_ids", batch.get("target_item_id"))

        if target is None:
            raise KeyError("Batch must contain target_item_ids or target_item_id.")

        target = target.long()

        recs = model.generate(
            batch,
            num_return_sequences=max_k,
        ).cpu()

        valid = target != pad_item_id
        valid &= target >= 0
        valid &= target < num_items

        if valid.sum() == 0:
            continue

        target = target[valid].cpu()
        recs = recs[valid]

        batch_size = target.size(0)
        total += batch_size

        for k in ks:
            topk = recs[:, :k]

            matches = topk.eq(target.unsqueeze(1))
            hit = matches.any(dim=1)

            hit_sum[k] += float(hit.float().sum().item())

            rank = matches.float().argmax(dim=1)
            ndcg = torch.zeros(batch_size, dtype=torch.float32)

            if hit.any():
                ndcg[hit] = 1.0 / torch.log2(rank[hit].float() + 2.0)

            ndcg_sum[k] += float(ndcg.sum().item())

            valid_recs = topk[
                (topk != pad_item_id)
                & (topk >= 0)
                & (topk < num_items)
            ]

            if valid_recs.numel() > 0:
                coverage_mask[k][valid_recs.unique()] = True

    if total == 0:
        raise RuntimeError("No valid test examples found.")

    catalog_size = max(num_items - 1, 1)

    metrics: dict[str, float] = {}

    for k in ks:
        metrics[f"hit_rate@{k}"] = hit_sum[k] / total
        metrics[f"ndcg@{k}"] = ndcg_sum[k] / total
        metrics[f"coverage@{k}"] = float(coverage_mask[k][1:].float().mean().item())

    metrics["num_test_examples"] = float(total)
    metrics["num_items"] = float(catalog_size)

    return metrics


In [ ]:
def run_one_model(
    *,
    model_name: str,
    dm,
) -> dict[str, float]:
    if model_name == "most_popular":
        model = fit_popularity_model(dm)
    elif model_name == "svd":
        model = fit_svd_model(dm)
    else:
        raise ValueError(f"Unknown model_name: {model_name}")

    metrics = evaluate_model(
        model,
        dm.test_dataloader(),
        num_items=dm.item_vocab_size,
        ks=(5, 10, 20),
        pad_item_id=0,
    )
    print(metrics)

    return metrics


def run_one_seed(
    dataset_name: str,
    dataset_cfg: dict[str, Any],
    seed: int,
) -> list[dict[str, Any]]:
    print(f"\n=== Dataset={dataset_name} | seed={seed} ===")

    set_seed(seed)

    data_cfg = make_data_config(dataset_cfg, seed)

    dm = AmazonReviewDataModule(data_cfg)
    dm.setup()

    print(json.dumps(dm.summary(), indent=2))

    rows: list[dict[str, Any]] = []

    for model_name in ("most_popular", "svd"):
        print(f"Running model: {model_name}")

        try:
            metrics = run_one_model(
                model_name=model_name,
                dm=dm,
            )

            row = {
                "dataset": dataset_name,
                "model": model_name,
                "seed": seed,
                **metrics,
            }

        except MemoryError as exc:
            print(f"Skipping {model_name} due to MemoryError: {exc}")

            row = {
                "dataset": dataset_name,
                "model": model_name,
                "seed": seed,
                "error": f"MemoryError: {exc}",
            }

        rows.append(row)

    return rows


def aggregate_rows(rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
    metric_keys = [
        "hit_rate@5",
        "hit_rate@10",
        "hit_rate@20",
        "ndcg@5",
        "ndcg@10",
        "ndcg@20",
        "coverage@5",
        "coverage@10",
        "coverage@20",
        "num_test_examples",
        "num_items",
    ]

    groups: dict[tuple[str, str], list[dict[str, Any]]] = {}

    for row in rows:
        if "error" in row:
            continue
        key = (row["dataset"], row["model"])
        groups.setdefault(key, []).append(row)

    summary_rows: list[dict[str, Any]] = []

    for (dataset, model), group_rows in groups.items():
        summary: dict[str, Any] = {
            "dataset": dataset,
            "model": model,
            "seed": "mean",
        }

        for metric in metric_keys:
            values = [
                float(row[metric])
                for row in group_rows
                if metric in row
            ]

            if not values:
                continue

            values_np = np.asarray(values, dtype=np.float64)
            summary[metric] = float(values_np.mean())
            summary[f"{metric}_std"] = float(values_np.std(ddof=0))

        summary_rows.append(summary)

    return summary_rows


def write_csv(path: Path, rows: list[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

    fieldnames: list[str] = []

    for row in rows:
        for key in row.keys():
            if key not in fieldnames:
                fieldnames.append(key)

    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


In [ ]:
output_dir = Path("../artifacts/metrics/amazon_baselines")
output_dir.mkdir(parents=True, exist_ok=True)

per_seed_rows: list[dict[str, Any]] = []

for dataset_name, dataset_cfg in AMAZON_DATASETS.items():
    for seed in SEEDS:
        per_seed_rows.extend(
            run_one_seed(
                dataset_name=dataset_name,
                dataset_cfg=dataset_cfg,
                seed=seed,
            )
        )

summary_rows = aggregate_rows(per_seed_rows)

write_csv(output_dir / "per_seed_metrics.csv", per_seed_rows)
write_csv(output_dir / "summary_metrics.csv", summary_rows)

print(f"\nSaved per-seed metrics to: {output_dir / 'per_seed_metrics.csv'}")
print(f"Saved summary metrics to: {output_dir / 'summary_metrics.csv'}")


| Dataset             | Model       |       HR@5 |     NDCG@5 |     Cov.@5 |      HR@10 |    NDCG@10 |    Cov.@10 |      HR@20 |    NDCG@20 |    Cov.@20 |
| ------------------- | ----------- | ---------: | ---------: | ---------: | ---------: | ---------: | ---------: | ---------: | ---------: | ---------: |
| Beauty              | MostPopular |     0.0008 |     0.0004 |     0.0004 |     0.0024 |     0.0009 |     0.0008 |     0.0083 |     0.0025 |     0.0017 |
| Beauty              | SVD         | **0.0158** | **0.0089** | **0.0628** | **0.0292** | **0.0132** | **0.0875** | **0.0479** | **0.0179** | **0.1267** |
| Musical Instruments | MostPopular |     0.0014 |     0.0005 |     0.0005 |     0.0219 |     0.0071 |     0.0010 |     0.0406 |     0.0119 |     0.0020 |
| Musical Instruments | SVD         | **0.0369** | **0.0189** | **0.0541** | **0.0542** | **0.0245** | **0.0813** | **0.0726** | **0.0291** | **0.1219** |
